# Test
<br>

> #### Gino Prasad
> #### 09/16/2024
<br>


In [10]:
import __main__ as main
import subprocess as sp
from tqdm import tqdm
notebook_name = 'difference_mapper'

In [11]:
import os, glob
import numpy as np
import pandas as pd
import subprocess as sp
from bs4 import BeautifulSoup
import re

In [12]:
html_pairs = sorted(glob.glob("/Users/ginoprasad/Job_Applications/web_crawler/dataset/UNFORMATTED/UNFORMATTED_v1.3_Adobe/unformatted_html_files/page_*"))
assert not len(html_pairs) % 2
html_pairs = np.stack([html_pairs[::2], html_pairs[1::2]], -1)

In [13]:
def get_score(a, b):
    if a == b:
        return 1
    if ' ' in a and ' ' in b:
        if a[:a.index(' ')] == b[:b.index(' ')]:
            return -1
    return float('-inf')

In [14]:
def get_last_column(s, t, indel_penalty):
    prev_scores = np.arange(len(s)+1) * indel_penalty
    for j, char_2 in enumerate(t, start=1):
        scores = np.ones(len(s)+1) * (j*indel_penalty)
        for i, char_1 in enumerate(s, start=1):
            score = get_score(char_1, char_2)
            scores[i] = max(score+prev_scores[i-1], 
                              indel_penalty+prev_scores[i], 
                              indel_penalty+scores[i-1])
        prev_scores = scores
    return prev_scores

In [15]:
def g_alignment(s, t, indel_penalty):
    if not t:
        return s, ['-']*len(s)
    elif not s:
        return ['-']*len(t), t
    
    mid = len(t) // 2
    
    max_ = float('-inf')
    decision = ''
    
    col_1 = get_last_column(s, t[:mid], indel_penalty)
    col_2 = get_last_column(s[::-1], t[mid+1:][::-1], indel_penalty)[::-1]

    for k in range(len(s), -1, -1):
        gap_score = indel_penalty + col_1[k] + col_2[k]
        if k < len(s):
            diag_score = get_score(s[k], t[mid]) + col_1[k] + col_2[k+1]
        else:
            diag_score = float('-inf')
        if gap_score > max_:
            max_ = gap_score
            decision = ('h', k)
        if diag_score > max_:
            max_ = diag_score
            decision = ('d', k)
    
    d, k = decision
    if d  == 'h':
        s_b, t_b = g_alignment(s[:k], t[:mid], indel_penalty)
        s_a, t_a = g_alignment(s[k:], t[mid+1:], indel_penalty)
        return s_b + ['-'] + s_a, t_b + t[mid:mid+1] + t_a
    else:
        s_b, t_b = g_alignment(s[:k], t[:mid], indel_penalty)
        s_a, t_a = g_alignment(s[k+1:], t[mid+1:], indel_penalty)
        return s_b + s[k:k+1] + s_a, t_b + t[mid:mid+1] + t_a

In [16]:
def format_html(a):
    return a.replace('\n', '').replace('<', '\n<').replace('"\n<', '"<').replace('\n</', '</').strip().split('\n')

In [17]:
# dataset_path = "/Users/ginoprasad/Job_Applications/web_crawler/aladdin/dataset"
dataset_path = "/Users/ginoprasad/Downloads"
if os.path.exists(f'{dataset_path}/diff.txt'):
    os.remove(f'{dataset_path}/diff.txt')
for a, b in tqdm(html_pairs):
    with open(a) as infile:
        s = format_html(infile.read())
    with open(b) as infile:
        t = format_html(infile.read())
    x, y = map(np.array, g_alignment(s, t, -1))
    assert len(x) == len(y)

    mask_raw = x != y
    i = j = 0
    with open(f'{dataset_path}/diff.txt', 'a') as outfile:
        outfile.write('-'*50 + '\n')
        outfile.write(f"{a}\n{b}\n\n")
        for xi, yj, index in zip(x[mask_raw], y[mask_raw], np.arange(len(mask_raw))[mask_raw]):
            outfile.write(f"{index-i if xi != '-' else xi},{index-j if yj != '-' else yj}\n{xi}\n{yj}\n\n")
            if xi == '-':
                i += 1
            if yj == '-':
                j += 1

100%|█████████| 5/5 [00:43<00:00,  8.71s/it]


In [9]:
for a, b in tqdm(html_pairs):
    sp.run(f"/usr/bin/open -a 'Google Chrome' '{a}'", shell=True)
    sp.run(f"/usr/bin/open -a 'Google Chrome' '{b}'", shell=True)
    input()

  0%|                 | 0/1 [00:00<?, ?it/s]

100%|█████████| 1/1 [00:17<00:00, 17.57s/it]


In [ ]:
# def g_alignment_high_ram(s, t, indel_penalty=-1, sub_penalty=-1):
#     matrix = np.zeros((len(s)+1, len(t)+1))
#     matrix[:,0] = np.arange(len(s)+1) * indel_penalty
#     matrix[0] = np.arange(len(t)+1) * indel_penalty
    
#     for i, char_1 in enumerate(s, start=1):
#         for j, char_2 in enumerate(t, start=1):
#             score = 1 if char_1 == char_2 else sub_penalty
#             matrix[i,j] = max(score+matrix[i-1,j-1], 
#                               indel_penalty+matrix[i-1,j], 
#                               indel_penalty+matrix[i,j-1])
#     print(matrix)
#     s_a = ''
#     t_a = ''
#     while i != 0 and j != 0:
#         score = 1 if s[i-1] == t[j-1] else sub_penalty
#         choice = np.argmax([score+matrix[i-1,j-1], 
#                           indel_penalty+matrix[i-1,j], 
#                           indel_penalty+matrix[i,j-1]])
     
#         if choice == 0:
#             s_a += s[i-1]
#             t_a += t[j-1]
#             i -= 1
#             j -= 1
#         elif choice == 1:
#             t_a += '-'
#             s_a += s[i-1]
#             i -= 1
#         else:
#             s_a += '-'
#             t_a += t[j-1]
#             j -= 1
#     s_a += '-' * j
#     s_a += s[:i][::-1]
#     t_a += '-' * i
#     t_a += t[:j][::-1]
    
#     return f"{s_a[::-1]}\n{t_a[::-1]}"